# Scoped Data Exploration

## Objective
Assess dataset coverage for the 11 selected PII categories,
inspect annotation quality, and prepare the data for
Regex baseline evaluation.

In [16]:
from pathlib import Path
import pandas as pd
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Current working directory:", Path.cwd())

Current working directory: <repository>\notebooks


In [17]:
from datasets import load_dataset
import pandas as pd
from collections import Counter

ds = load_dataset("mabahboh/sitr-arabic-pii")

print(ds)
print("Available splits:", list(ds.keys()))

'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/datasets/mabahboh/sitr-arabic-pii/resolve/9635384c7e241c57258ac8db73120deba3f34223/sitr-arabic-pii.py
Retrying in 1s [Retry 1/5].
'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/datasets/mabahboh/sitr-arabic-pii/resolve/9635384c7e241c57258ac8db73120deba3f34223/sitr-arabic-pii.py
Retrying in 2s [Retry 2/5].
'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/datasets/mabahboh/sitr-arabic-pii/resolve/9635384c7e241c57258ac8db73120deba3f34223/sitr-arabic-pii.py
Retrying in 4s [Retry 3/5].
'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/datasets/mabahboh/sitr-arabic-pii/resolve/

DatasetDict({
    train: Dataset({
        features: ['text', 'entities', 'type', 'secret_text'],
        num_rows: 14288
    })
    validation: Dataset({
        features: ['text', 'entities', 'type', 'secret_text'],
        num_rows: 1818
    })
    test: Dataset({
        features: ['text', 'entities', 'type', 'secret_text'],
        num_rows: 1793
    })
})
Available splits: ['train', 'validation', 'test']


In [18]:
entity_counts = Counter(
    entity["label"]
    for entities in ds["train"]["entities"]
    for entity in entities
)

label_df = pd.DataFrame(
    entity_counts.items(),
    columns=["entity_type", "occurrences"]
).sort_values("occurrences", ascending=False)

print("Number of unique entity types:", len(label_df))
display(label_df)

Number of unique entity types: 43


,entity_type,occurrences
4,private_phone,3818
18,national_id,3123
26,private_email,2385
6,employee_id,1819
2,private_date,1753
12,iqama,1723
16,account_number,1509
1,insurance_policy,1464
13,national_address,1299
34,medical_record_number,1191


In [19]:
target_categories = [
    "private_email",
    "private_phone",
    "national_id",
    "iqama",
    "api_key",
    "access_token",
    "password",
    "credit_card",
    "account_number",
    "private_address",
    "national_address",
]

coverage = []

for category in target_categories:
    count = entity_counts.get(category, 0)

    coverage.append({
        "category": category,
        "occurrences": count,
        "present_in_train": count > 0
    })

coverage_df = pd.DataFrame(coverage)

display(coverage_df)

,category,occurrences,present_in_train
0,private_email,2385,True
1,private_phone,3818,True
2,national_id,3123,True
3,iqama,1723,True
4,api_key,252,True
5,access_token,239,True
6,password,455,True
7,credit_card,704,True
8,account_number,1509,True
9,private_address,914,True


In [20]:
def check_entity_offsets(dataset):
    total = 0
    invalid = 0
    examples = []

    for row_idx, row in enumerate(dataset):
        text = row["text"]

        for entity in row["entities"]:
            total += 1

            start = entity["start"]
            end = entity["end"]

            valid = (
                isinstance(start, int)
                and isinstance(end, int)
                and 0 <= start < end <= len(text)
            )

            if not valid:
                invalid += 1

                if len(examples) < 5:
                    examples.append({
                        "row": row_idx,
                        "entity": entity,
                        "text_length": len(text),
                    })

    return {
        "total_entities": total,
        "invalid_offsets": invalid,
        "valid_offsets": total - invalid,
        "examples": examples,
    }


offset_results = check_entity_offsets(ds["train"])

print("Total entities:", offset_results["total_entities"])
print("Valid offsets:", offset_results["valid_offsets"])
print("Invalid offsets:", offset_results["invalid_offsets"])
print("Invalid examples:", offset_results["examples"])

Total entities: 32564
Valid offsets: 32564
Invalid offsets: 0
Invalid examples: []


In [21]:
mismatch_examples = []
total_entities = 0
mismatch_count = 0

for row_idx, row in enumerate(ds["train"]):
    text = row["text"]

    for entity in row["entities"]:
        total_entities += 1

        extracted = text[entity["start"]:entity["end"]]

        # Compare with the original entity value if available
        expected = entity.get("text", entity.get("value"))

        if expected is not None and extracted != expected:
            mismatch_count += 1

            if len(mismatch_examples) < 10:
                mismatch_examples.append({
                    "row": row_idx,
                    "label": entity["label"],
                    "expected": expected,
                    "extracted": extracted,
                    "start": entity["start"],
                    "end": entity["end"],
                })

print("Total entities:", total_entities)
print("Mismatches:", mismatch_count)
print("Examples:")

for item in mismatch_examples:
    print(item)

Total entities: 32564
Mismatches: 0
Examples:


In [22]:
target_categories = [
    "private_email",
    "private_phone",
    "national_id",
    "iqama",
    "api_key",
    "access_token",
    "password",
    "credit_card",
    "account_number",
    "private_address",
    "national_address",
]

rows = []

for split_name, dataset in ds.items():
    counts = Counter(
        entity["label"]
        for sample in dataset
        for entity in sample["entities"]
    )

    for category in target_categories:
        rows.append({
            "split": split_name,
            "category": category,
            "occurrences": counts.get(category, 0),
            "present": counts.get(category, 0) > 0,
        })

coverage_by_split = pd.DataFrame(rows)

display(
    coverage_by_split.pivot(
        index="category",
        columns="split",
        values="occurrences"
    )
)

split,test,train,validation
category,,,
access_token,40,239,29
account_number,212,1509,195
api_key,32,252,23
credit_card,122,704,94
iqama,188,1723,213
national_address,156,1299,148
national_id,390,3123,441
password,62,455,49
private_address,93,914,101


In [23]:
import pandas as pd

target_categories = [
    "private_email",
    "private_phone",
    "national_id",
    "iqama",
    "api_key",
    "access_token",
    "password",
    "credit_card",
    "account_number",
    "private_address",
    "national_address",
]

quality_results = []

for split_name, dataset in ds.items():
    empty_texts = 0
    empty_entities = 0
    no_target_entity = 0

    for row in dataset:
        text = row.get("text")
        entities = row.get("entities") or []

        if not isinstance(text, str) or not text.strip():
            empty_texts += 1

        if not entities:
            empty_entities += 1

        labels = {entity["label"] for entity in entities}

        if not labels.intersection(target_categories):
            no_target_entity += 1

    quality_results.append({
        "split": split_name,
        "total_records": len(dataset),
        "empty_texts": empty_texts,
        "empty_entities": empty_entities,
        "records_without_target_entity": no_target_entity,
    })

quality_df = pd.DataFrame(quality_results)
display(quality_df)

,split,total_records,empty_texts,empty_entities,records_without_target_entity
0,train,14288,0,2867,7611
1,validation,1818,0,373,981
2,test,1793,0,380,969


In [24]:
# Inspect records with no annotated entities in the test split

empty_entity_examples = [
    row
    for row in ds["test"]
    if not row["entities"]
][:5]

for i, row in enumerate(empty_entity_examples, start=1):
    print(f"\n--- Example {i} ---")
    print("Type:", row["type"])
    print("Text:", row["text"][:500])


--- Example 1 ---
Type: negative
Text: اقترح تحسين صياغة التقرير مع الحفاظ على محتواه

تقرير لجنة إدارية مقترح

بناء على توجيهات الجهة المُشرِفَه اجتمعت اللجنة لمتابعة تدارك الملاحظات الوارده في تقرير الأداء للربع المنقضي

الايجابيات
- الالتزام بشروط الصحه والسلامه داخل المبنى
- انتظام سير العمل لدى الفرق
- تخفيض زمن الاستجابة للطلبات

السلبيات
- ضعف التوثيق الورقي في بعض الأقسام
- تأخر رفع التقارير الدوريه
- نقص عدد الفنيين في الورديه المسائيه

التوصيات النهائية
1 استكمال قاعدة البيانات الالكترونيه
2 تعيين فرق دعم إضافيه
3 إلزام رؤس

--- Example 2 ---
Type: negative
Text: اكتب خطاب اعتذار رسمي للعميل بسبب تأخير تسليم الطلب.

شركة الوفاء للتوريدات
خطاب اعتذار

العميل: مؤسسة البناء الحديثة
رقم الطلب: PO-2024-773
تاريخ الطلب: 2024/03/01
تاريخ التسليم المتفق: 2024/03/20
تاريخ التسليم الفعلي: 2024/03/28
قيمة الطلب: 45000 ريال سعودي

السبب:
تأخر التوريد من المصنع الخارجي بسبب عطل في خط الإنتاج، مما أدى إلى تأجيل الشحن لمدة أسبوع.

الإجراء التعويضي:
تمنح الشركة خصم 5% على قيمة الطلب المقبل 

In [25]:
# Inspect record types across all dataset splits

type_summary = []

for split_name, dataset in ds.items():
    counts = Counter(row["type"] for row in dataset)

    for record_type, count in counts.items():
        type_summary.append({
            "split": split_name,
            "type": record_type,
            "count": count,
        })

type_summary_df = pd.DataFrame(type_summary)

display(
    type_summary_df.pivot(
        index="type",
        columns="split",
        values="count"
    ).fillna(0).astype(int)
)

split,test,train,validation
type,,,
access_token,4,29,4
account_number,28,291,25
api_key,8,38,6
bahrain_cpr,10,93,17
boundary_number,18,186,33
building_number,24,173,29
card_cvv,9,94,12
card_pin,16,106,11
commercial_registration,24,266,37


In [26]:
# Check consistency between record type and entity annotations

consistency_results = []

for split_name, dataset in ds.items():
    negative_with_entities = 0
    non_negative_without_entities = 0

    for row in dataset:
        has_entities = bool(row["entities"])

        if row["type"] == "negative" and has_entities:
            negative_with_entities += 1

        elif row["type"] != "negative" and not has_entities:
            non_negative_without_entities += 1

    consistency_results.append({
        "split": split_name,
        "negative_with_entities": negative_with_entities,
        "non_negative_without_entities": non_negative_without_entities,
    })

consistency_df = pd.DataFrame(consistency_results)
display(consistency_df)

,split,negative_with_entities,non_negative_without_entities
0,train,0,0
1,validation,0,0
2,test,0,0


In [27]:
# Check exact text overlap between dataset splits

split_texts = {
    split_name: {
        row["text"].strip()
        for row in dataset
        if isinstance(row["text"], str) and row["text"].strip()
    }
    for split_name, dataset in ds.items()
}

pairs = [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]

for split_a, split_b in pairs:
    overlap = split_texts[split_a] & split_texts[split_b]

    print(f"{split_a} vs {split_b}")
    print("Exact duplicate texts:", len(overlap))
    print("-" * 35)

train vs validation
Exact duplicate texts: 0
-----------------------------------
train vs test
Exact duplicate texts: 0
-----------------------------------
validation vs test
Exact duplicate texts: 0
-----------------------------------


In [28]:
# Count records containing each target category in the test split

target_categories = [
    "private_email",
    "private_phone",
    "national_id",
    "iqama",
    "api_key",
    "access_token",
    "password",
    "credit_card",
    "account_number",
    "private_address",
    "national_address",
]

test_record_counts = Counter()

for row in ds["test"]:
    labels = {entity["label"] for entity in row["entities"]}

    for category in target_categories:
        if category in labels:
            test_record_counts[category] += 1

test_records_df = pd.DataFrame([
    {
        "category": category,
        "entity_occurrences": entity_counts.get(category, 0),
        "test_records": test_record_counts[category],
    }
    for category in target_categories
])

display(
    test_records_df.sort_values("test_records", ascending=False)
)

,category,entity_occurrences,test_records
1,private_phone,3818,312
2,national_id,3123,254
0,private_email,2385,182
3,iqama,1723,132
8,account_number,1509,128
10,national_address,1299,112
7,credit_card,704,87
9,private_address,914,80
6,password,455,52
5,access_token,239,32


In [29]:
# Verify that each positive record has an entity matching its type

mismatches = []

for split_name, dataset in ds.items():
    for row_idx, row in enumerate(dataset):
        record_type = row["type"]
        labels = {entity["label"] for entity in row["entities"]}

        if record_type != "negative" and record_type not in labels:
            mismatches.append({
                "split": split_name,
                "row": row_idx,
                "type": record_type,
                "entity_labels": sorted(labels),
            })

print("Records checked:", sum(len(dataset) for dataset in ds.values()))
print("Type-label mismatches:", len(mismatches))

for item in mismatches[:10]:
    print(item)

Records checked: 17899
Type-label mismatches: 0


In [30]:

# Analyze records containing both target and non-target entity categories

target_set = set(target_categories)

mixed_label_results = []

for split_name, dataset in ds.items():
    target_only = 0
    non_target_only = 0
    mixed_records = 0
    no_entities = 0

    for row in dataset:
        labels = {
            entity["label"]
            for entity in row["entities"]
        }

        if not labels:
            no_entities += 1
            continue

        has_target = bool(labels & target_set)
        has_non_target = bool(labels - target_set)

        if has_target and has_non_target:
            mixed_records += 1
        elif has_target:
            target_only += 1
        else:
            non_target_only += 1

    mixed_label_results.append({
        "split": split_name,
        "target_only_records": target_only,
        "non_target_only_records": non_target_only,
        "mixed_label_records": mixed_records,
        "records_without_entities": no_entities,
    })

mixed_label_df = pd.DataFrame(mixed_label_results)

display(mixed_label_df)


,split,target_only_records,non_target_only_records,mixed_label_records,records_without_entities
0,train,2818,4744,3859,2867
1,validation,360,608,477,373
2,test,354,589,470,380


In [31]:

# Calculate total target entities and unique records per split

target_set = set(target_categories)

total_results = []

for split_name, dataset in ds.items():
    total_target_entities = 0
    target_records = 0

    for row in dataset:
        target_entities = [
            entity for entity in row["entities"]
            if entity["label"] in target_set
        ]

        total_target_entities += len(target_entities)

        if target_entities:
            target_records += 1

    total_results.append({
        "split": split_name,
        "total_records": len(dataset),
        "target_records": target_records,
        "total_target_entities": total_target_entities,
    })

total_df = pd.DataFrame(total_results)
display(total_df)


,split,total_records,target_records,total_target_entities
0,train,14288,6677,16421
1,validation,1818,837,2157
2,test,1793,824,2079


In [32]:

target_set = set(target_categories)

for split_name, dataset in ds.items():
    target_samples = sum(
        1
        for row in dataset
        if any(
            entity["label"] in target_set
            for entity in row["entities"]
        )
    )

    print(f"{split_name}: {target_samples} samples")

print("Total dataset samples:", sum(len(dataset) for dataset in ds.values()))


train: 6677 samples
validation: 837 samples
test: 824 samples
Total dataset samples: 17899


In [36]:

# Cell 20: Inspect negative samples

negative_results = []

for split_name, dataset in ds.items():
    empty_rows = [
        {
            "split": split_name,
            "row": row_idx,
            "type": row["type"],
            "text": row["text"],
        }
        for row_idx, row in enumerate(dataset)
        if not row["entities"]
    ]

    negative_results.append({
        "split": split_name,
        "records_without_entities": len(empty_rows),
    })

    print(f"\n--- {split_name}: examples without entities ---")

    for sample in empty_rows[:3]:
        print({
            "row": sample["row"],
            "type": sample["type"],
            "text": sample["text"][:250],
        })

display(pd.DataFrame(negative_results))



--- train: examples without entities ---
{'row': 2, 'type': 'negative', 'text': 'Simplify the following text for me.\n\nعرض سعر\nالسيد/ العميل الكريم\nالتاريخ: 2024/02/20\n\nنقدم لكم عرض سعر لخدمات الصيانه السنويه:\n\n1. صيانه مكيفات: 2500 ريال\n2. صيانه مصاعد: 4000 ريال\n\nالاجمالي: 6500 ريال\nغير شامل ضريبههههه القيمه المضافه.\n\nرقم العرض: '}
{'row': 7, 'type': 'negative', 'text': 'هذا نص مترجم وارغب منك تصحيح الاخطاء الاملائية والنحوية فيه مع مراجعة علامات الترقيم\n\nمذكرة تفاهم\n\nبين وزارة الاتصالات وتقنية المعلومات\nو شركة الافق لتقنية المعلومات\n\nالموضوع التعاون في تطوير الخدمات الرقمية\nالتاريخ 2025/01/20م\n\nبناء على الرغبة المشت'}
{'row': 8, 'type': 'negative', 'text': 'شلون اضيف مستفيد جديد اذا التحويل دايم يفشل ما ادري شنو اسوي يطلععع لي خطا كل مره؟'}

--- validation: examples without entities ---
{'row': 3, 'type': 'negative', 'text': 'Compare these two versions and highlight any differences in dates, names, or numbers.\n\nالمستند الاول:\n\nسيره ذاتيه\n\nالبيانات الشخصي

,split,records_without_entities
0,train,2867
1,validation,373
2,test,380
